Imports and Engine Initialization

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add parent directory to path to import our retrieval modules
sys.path.append("..")
from src.retrieval.vector_search import VectorSearchEngine
from src.retrieval.bm25_search import BM25Okapi
from src.retrieval.bm25_search import BM25SearchEngine
from src.retrieval.collaborative_filtering import CollaborativeFilteringEngine

print("Initializing all three retrieval engines...")
vector_engine = VectorSearchEngine(db_dir="../data/chroma_db")
bm25_engine = BM25SearchEngine(csv_path="../data/books_cleaned.csv")
cf_engine = CollaborativeFilteringEngine(csv_path="../data/books_cleaned.csv")

print("All engines initialized successfully.")


/Users/simranchaudhary/Documents/ShelfKnows/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Initializing all three retrieval engines...
Initializing Embedding Model ('all-MiniLM-L6-v2')...


/Users/simranchaudhary/Documents/ShelfKnows/venv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Connecting to local ChromaDB vector store at '../data/chroma_db'...
 Building BM25 Sparse Index from '../data/books_cleaned.csv'...
 BM25 Index successfully built for 50000 books in 0.86 seconds!
 Initializing Collaborative Filtering Engine from '../data/books_cleaned.csv'...
 Collaborative Filtering Model trained across 500 books in 1.25 seconds!
All engines initialized successfully.


Querying All Three Engines Independently

In [2]:
query_text = "The Hunger Games dystopian survival"
top_k = 10

print(f"Running individual searches for: '{query_text}'")

# 1. Dense Vector Search
vector_results = vector_engine.search(query_text, top_k=top_k)

# 2. Sparse BM25 Search
bm25_results = bm25_engine.search(query_text, top_k=top_k)

# 3. Collaborative Filtering (using top matched book title)
cf_results = []
if len(bm25_results) > 0:
    seed_title = bm25_results[0]['title']
    cf_results = cf_engine.recommend(seed_title, top_k=top_k)

print(f"Retrieved: {len(vector_results)} Vector results, {len(bm25_results)} BM25 results, {len(cf_results)} CF results.")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Running individual searches for: 'The Hunger Games dystopian survival'
⚡ Vector Search for 'The Hunger Games dystopian survival' returned 10 matches in 166.09 ms
⚡ BM25 Search for 'The Hunger Games dystopian survival' returned 10 matches in 49.00 ms
 Book 'The Hunger Games (The Hunger Games, #1)' not found in top interaction catalog.
Retrieved: 10 Vector results, 10 BM25 results, 0 CF results.


/Users/simranchaudhary/Documents/ShelfKnows/notebook/../src/retrieval/collaborative_filtering.py:60: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = self.top_books[self.top_books['title'].str.contains(book_title, case=False, na=False)]


Reciprocal Rank Fusion (RRF) Calculation Function

In [3]:
def compute_rrf(list_of_ranked_lists, k=60):
    """
    Computes Reciprocal Rank Fusion scores across multiple ranked lists.
    Formula: RRF_Score = sum( 1 / (k + rank) )
    """
    rrf_scores = {}
    book_details = {}

    for ranked_list in list_of_ranked_lists:
        for rank, book in enumerate(ranked_list, start=1):
            book_id = book.get('book_id')
            if not book_id:
                continue

            # Calculate reciprocal rank contribution
            reciprocal_rank = 1.0 / (k + rank)
            
            if book_id not in rrf_scores:
                rrf_scores[book_id] = 0.0
                book_details[book_id] = book
                
            rrf_scores[book_id] += reciprocal_rank

    # Sort books by combined RRF score in descending order
    sorted_book_ids = sorted(rrf_scores.keys(), key=lambda bid: rrf_scores[bid], reverse=True)

    fused_results = []
    for bid in sorted_book_ids:
        entry = dict(book_details[bid])
        entry['rrf_score'] = round(rrf_scores[bid], 5)
        fused_results.append(entry)

    return fused_results

# Execute RRF fusion
fused_candidates = compute_rrf([vector_results, bm25_results, cf_results], k=60)
print(f"Successfully fused {len(fused_candidates)} unique candidate books.")


Successfully fused 18 unique candidate books.


Displaying the Final Top-10 Hybrid Recommendations

In [4]:
# Convert to DataFrame for clean inspection
df_fused = pd.DataFrame(fused_candidates).head(10)

# Deduplicate by title to ensure distinct books
df_fused = df_fused.drop_duplicates(subset=['title']).head(10)

print("Top 10 Hybrid Recommendations (Dense Vector + BM25 + Collaborative Filtering):")
display(df_fused[['title', 'authors', 'genres', 'average_rating', 'rrf_score']])


Top 10 Hybrid Recommendations (Dense Vector + BM25 + Collaborative Filtering):


,title,authors,genres,average_rating,rrf_score
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,General Fiction,4.34,0.03279
1,The World of the Hunger Games (Hunger Games Tr...,Kate Egan,History & Biography,4.48,0.03062
3,The Hunger Games Tribute Guide,Emily Seife,General Fiction,4.40,0.01587
5,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,General Fiction,4.49,0.01562
7,The Hunger Games: Official Illustrated Movie C...,Kate Egan,General Fiction,4.51,0.01538
9,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,General Fiction,4.30,0.01515
